# Binding Preservation Analysis

Evaluates whether generated/optimized antibody sequences preserve binding capability using:
1. **AlphaFold-Multimer** — predicted interface pTM (ipTM) and pLDDT scores
2. **Sequence identity** — CDR3 conservation between original and optimized sequences

This addresses the core concern: do developability-optimized sequences still bind their targets?

In [ ]:
!pip install colabfold biopython matplotlib pandas numpy 2>/dev/null || echo "Install ColabFold separately if needed"

In [ ]:
import os, re, json, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_DIR = Path("results/binding_preservation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GDPA1_DATA_PATH = "GDPa1_v1.2_20250814.csv"
VH_COL = "vh_protein_sequence"
VL_COL = "vl_protein_sequence"

# Number of antibodies to evaluate (AF2 is expensive)
N_EVAL = 10
SEED = 42
np.random.seed(SEED)

## 1. Select Evaluation Subset

Pick approved antibodies with known targets for structural validation.

In [ ]:
df = pd.read_csv(GDPA1_DATA_PATH)

approved = df[df["highest_clinical_trial_asof_feb2025"] == "Approved"].copy()
print(f"Total approved antibodies: {len(approved)}")

# Check for target antigen column
target_candidates = [c for c in df.columns if "target" in c.lower() or "antigen" in c.lower()]
print(f"Potential target columns: {target_candidates}")

# Sample a subset for evaluation
eval_subset = approved.sample(n=min(N_EVAL, len(approved)), random_state=SEED).reset_index(drop=True)
print(f"\nSelected {len(eval_subset)} antibodies for evaluation")
eval_subset[[VH_COL, VL_COL]].head()

## Prepare FASTA Files for AlphaFold-Multimer

Each antibody is formatted as a VH:VL heterodimer for structure prediction.

In [ ]:
fasta_dir = OUTPUT_DIR / "fasta_inputs"
fasta_dir.mkdir(exist_ok=True)

fasta_paths = []
for i, row in eval_subset.iterrows():
    vh = str(row[VH_COL]).strip()
    vl = str(row[VL_COL]).strip()
    name = f"ab_{i:03d}"

    # AlphaFold-Multimer format: separate chains with ':'
    fasta_content = f">{name}_VH_VL\n{vh}:{vl}\n"

    fasta_path = fasta_dir / f"{name}.fasta"
    with open(fasta_path, "w") as f:
        f.write(fasta_content)
    fasta_paths.append(fasta_path)

print(f"Wrote {len(fasta_paths)} FASTA files to {fasta_dir}")
print(f"Example:\n{open(fasta_paths[0]).read()}")

## Run AlphaFold-Multimer (or ColabFold)

In [ ]:
af_output_dir = OUTPUT_DIR / "af2_outputs"
af_output_dir.mkdir(exist_ok=True)

# Try running ColabFold locally
colabfold_available = False
try:
    result = subprocess.run(["colabfold_batch", "--help"], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        colabfold_available = True
except (FileNotFoundError, subprocess.TimeoutExpired):
    pass

if colabfold_available:
    print("ColabFold found. Running predictions...")
    for fasta_path in fasta_paths:
        name = fasta_path.stem
        out_dir = af_output_dir / name
        out_dir.mkdir(exist_ok=True)
        cmd = [
            "colabfold_batch",
            str(fasta_path),
            str(out_dir),
            "--num-recycle", "3",
            "--model-type", "alphafold2_multimer_v3",
        ]
        print(f"  Running: {name}")
        subprocess.run(cmd, capture_output=True, text=True)
else:
    print("ColabFold not found locally.")
    print("Options:")
    print("  1. Install: pip install colabfold[alphafold]")
    print("  2. Use Google Colab: https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb")
    print("  3. Place pre-computed results in:", af_output_dir)
    print("\nGenerating submission script...")

    script = f"""#!/bin/bash
# Run ColabFold on all FASTA files
for fasta in {fasta_dir}/*.fasta; do
    name=$(basename "$fasta" .fasta)
    colabfold_batch "$fasta" {af_output_dir}/"$name" \\
        --num-recycle 3 \\
        --model-type alphafold2_multimer_v3
done
"""
    script_path = OUTPUT_DIR / "run_colabfold.sh"
    with open(script_path, "w") as f:
        f.write(script)
    print(f"Saved script to: {script_path}")

## Parse AlphaFold-Multimer Results

In [ ]:
def parse_af2_scores(output_dir: Path) -> list:
    """Parse pTM, ipTM, and pLDDT from ColabFold JSON outputs."""
    records = []
    for name_dir in sorted(output_dir.iterdir()):
        if not name_dir.is_dir():
            continue
        name = name_dir.name
        # ColabFold outputs: *_scores_rank_001_*.json
        score_files = sorted(name_dir.glob("*scores_rank_001*.json"))
        if not score_files:
            score_files = sorted(name_dir.glob("*scores*.json"))
        if not score_files:
            continue

        with open(score_files[0]) as f:
            scores = json.load(f)

        record = {
            "antibody": name,
            "ptm": scores.get("ptm", np.nan),
            "iptm": scores.get("iptm", np.nan),
            "plddt_mean": np.mean(scores.get("plddt", [np.nan])),
        }
        # Combined confidence: 0.8*ipTM + 0.2*pTM (AF2-Multimer ranking score)
        if not np.isnan(record["iptm"]) and not np.isnan(record["ptm"]):
            record["ranking_confidence"] = 0.8 * record["iptm"] + 0.2 * record["ptm"]
        else:
            record["ranking_confidence"] = np.nan
        records.append(record)

    return records

af2_results = parse_af2_scores(af_output_dir)

if af2_results:
    af2_df = pd.DataFrame(af2_results)
    print(f"Parsed {len(af2_df)} AF2 predictions")
    print(af2_df.to_string(index=False))
    af2_df.to_csv(OUTPUT_DIR / "af2_scores.csv", index=False)
else:
    print("No AF2 results found yet. Run ColabFold first (see cell above).")
    print("Creating placeholder with expected columns for downstream cells...")
    af2_df = pd.DataFrame(columns=["antibody", "ptm", "iptm", "plddt_mean", "ranking_confidence"])

## Visualize Binding Confidence

In [ ]:
if len(af2_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for i, (col, label, thresh) in enumerate([
        ("iptm", "Interface pTM (ipTM)", 0.6),
        ("plddt_mean", "Mean pLDDT", 70),
        ("ranking_confidence", "Ranking Confidence\n(0.8·ipTM + 0.2·pTM)", 0.6),
    ]):
        ax = axes[i]
        vals = af2_df[col].dropna()
        if len(vals) == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            continue
        bars = ax.bar(range(len(vals)), vals.values, color="#2d6a9f", edgecolor="black", linewidth=0.5)
        ax.axhline(thresh, color="red", linestyle="--", linewidth=1, label=f"Threshold={thresh}")
        ax.set_xlabel("Antibody")
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.legend(fontsize=8)

        # Color bars below threshold red
        for bar, v in zip(bars, vals.values):
            if v < thresh:
                bar.set_color("#d9534f")

    plt.suptitle("AlphaFold-Multimer Binding Confidence", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "af2_binding_confidence.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Summary statistics
    print("\nSummary:")
    for col, thresh, label in [("iptm", 0.6, "ipTM"), ("plddt_mean", 70, "pLDDT"), ("ranking_confidence", 0.6, "Ranking")]:
        vals = af2_df[col].dropna()
        if len(vals) > 0:
            n_pass = (vals >= thresh).sum()
            print(f"  {label}: mean={vals.mean():.3f}, {n_pass}/{len(vals)} pass threshold ({thresh})")
else:
    print("No AF2 results to visualize. Run prediction first.")

## Sequence Conservation Analysis

Even without AF2, we can check how much CDR3 diversity exists in the dataset as a baseline for any future optimization comparisons.

In [ ]:
def extract_cdr3(seq, chain="H"):
    seq = str(seq).upper()
    if chain == "H":
        m = re.search(r'C([A-Z]{3,30})W[GS]', seq)
    else:
        m = re.search(r'C([A-Z]{3,20})F[GS]', seq)
    return m.group(1) if m else None

# Extract CDR3s from evaluation subset
eval_subset["cdr3h"] = eval_subset[VH_COL].apply(lambda s: extract_cdr3(s, "H"))
eval_subset["cdr3l"] = eval_subset[VL_COL].apply(lambda s: extract_cdr3(s, "L"))

print("CDR3-H sequences:")
for i, row in eval_subset.iterrows():
    print(f"  ab_{i:03d}: {row['cdr3h']} (len={len(row['cdr3h']) if row['cdr3h'] else 'N/A'})")

print(f"\nCDR3-L sequences:")
for i, row in eval_subset.iterrows():
    print(f"  ab_{i:03d}: {row['cdr3l']} (len={len(row['cdr3l']) if row['cdr3l'] else 'N/A'})")

# CDR3 length stats
for chain, col in [("H", "cdr3h"), ("L", "cdr3l")]:
    lengths = eval_subset[col].dropna().apply(len)
    print(f"\nCDR3-{chain} length: mean={lengths.mean():.1f}, std={lengths.std():.1f}, "
          f"range=[{lengths.min()}, {lengths.max()}]")

## Summary

In [ ]:
summary = {
    "n_evaluated": len(eval_subset),
    "af2_completed": len(af2_df) if len(af2_df) > 0 else 0,
}

if len(af2_df) > 0:
    summary["iptm_mean"] = af2_df["iptm"].mean()
    summary["iptm_pass_rate"] = (af2_df["iptm"] >= 0.6).mean()
    summary["plddt_mean"] = af2_df["plddt_mean"].mean()
    summary["ranking_confidence_mean"] = af2_df["ranking_confidence"].mean()

summary_text = f"""
Binding Preservation Summary:
- Evaluated {summary['n_evaluated']} approved antibodies
- AF2-Multimer predictions completed: {summary['af2_completed']}
"""

if summary["af2_completed"] > 0:
    summary_text += f"""- Mean ipTM: {summary['iptm_mean']:.3f} (pass rate ≥0.6: {summary['iptm_pass_rate']:.1%})
- Mean pLDDT: {summary['plddt_mean']:.1f}
- Mean ranking confidence: {summary['ranking_confidence_mean']:.3f}

Interpretation: ipTM ≥ 0.6 indicates confident interface prediction.
High scores suggest the VH:VL pairing forms a structurally plausible Fv domain.
"""
else:
    summary_text += """
- AF2 results pending. Run ColabFold using the generated FASTA files and script.
- FASTA files are ready for submission in results/binding_preservation/fasta_inputs/
"""

print(summary_text)

with open(OUTPUT_DIR / "binding_preservation_summary.txt", "w") as f:
    f.write(summary_text)

pd.DataFrame([summary]).to_csv(OUTPUT_DIR / "binding_preservation_metrics.csv", index=False)